In [ ]:
from google.cloud import bigquery
import warnings
import pandas as pd
from datasets import Dataset
from utils.hub_card import push_dataset_card
warnings.filterwarnings(action='ignore', message="Your application has authenticated using end user")

client = bigquery.Client(project="ads-599-final")

In [ ]:
# Pre-arrival medication reconciliation from ed.medrecon.
# Medications the patient was already taking before the ED visit.
# Treated as static/baseline state — not time-varying actions.
# Multiple rows per visit: one per drug per ETC therapeutic class group.
# etcdescription groups drugs by class (e.g. 'Beta Blockers', 'ACE Inhibitors').
# Deduplication applied: same patient + stay + drug identity collapsed to one row.

query_medrecon = """
WITH cohort_subjects AS (
  SELECT ed_stay_id, subject_id
  FROM `ads-599-final.rl_project.cohort_base`
)
SELECT
  m.stay_id       AS ed_stay_id,
  m.subject_id,
  m.charttime,
  m.name,
  m.gsn,
  m.ndc,
  m.etccode,
  m.etcdescription
FROM `physionet-data.mimiciv_ed.medrecon` m
INNER JOIN cohort_subjects cs ON m.stay_id = cs.ed_stay_id
ORDER BY m.stay_id, m.name
"""

print("Running medrecon query...")
df_medrecon = client.query(query_medrecon).to_dataframe()
print(f"Shape before dedup: {df_medrecon.shape}")
print(f"Unique ED visits: {df_medrecon['ed_stay_id'].nunique():,}")
df_medrecon.head()

In [ ]:
# Deduplication: same patient + stay + drug identity can appear multiple times
# due to multiple GSN ontology group mappings. Collapse to one row per drug per visit.
dupes = df_medrecon.duplicated(subset=['subject_id', 'ed_stay_id', 'name', 'gsn', 'ndc', 'etccode']).sum()
print(f"Duplicate rows removed: {dupes:,}")
df_medrecon.drop_duplicates(subset=['subject_id', 'ed_stay_id', 'name', 'gsn', 'ndc', 'etccode'], inplace=True, ignore_index=True)
print(f"Shape after dedup: {df_medrecon.shape}")

In [ ]:
DESCRIPTION = (
    "Medication reconciliation records from ed.medrecon — medications the patient was taking "
    "prior to ED arrival. One row per medication per ED visit. "
    "Represents pre-arrival medication state, not actions taken during the visit."
)

ds = Dataset.from_pandas(df_medrecon, split='medrecon')
ds.push_to_hub("ADS599-Capstone/raw_data", config_name="medrecon", data_dir="pre_arrival_meds")
push_dataset_card("ADS599-Capstone/raw_data", config_name="medrecon", split="medrecon", data_dir="pre_arrival_meds", description=DESCRIPTION)
print("Pushed medrecon to HuggingFace Hub.")

In [ ]:
# 1. Map ETCDESCRIPTION to shared 22-class vocabulary
#
# Raw etcdescription has 1,199 unique drug class labels — far too many to encode
# directly as model features (each would be a near-zero binary flag for most visits).
# Collapsed to 22 clinically meaningful groups shared with the ed_only_meds table.
#
# Why 22 groups:
#   ed_only_meds has no etcdescription column; it uses regex on raw drug name strings.
#   Without harmonization, recon_* features (1,199 cols) and edmeds_* features (22 cols)
#   would occupy different feature spaces and be incomparable in the RL state vector.
#   22 groups cover all major pharmacological categories relevant to ED acuity while
#   keeping the state vector tractable (~26 medication columns vs 1,202 without this).
#   Mapping uses substring match on etcdescription (case-insensitive, first match wins).
#   Unmatched classes fall through to 'Other'.

ETCDESC_TO_CLASS = [
    ('opioid',                         'Analgesic - Opioid/NSAID'),
    ('analgesic opioid',               'Analgesic - Opioid/NSAID'),
    ('narcotic',                       'Analgesic - Opioid/NSAID'),
    ('nsaid',                          'Analgesic - NSAID'),
    ('salicylate analgesic',           'Analgesic - NSAID'),
    ('non-opioid',                     'Analgesic - Acetaminophen'),
    ('acetaminophen',                  'Analgesic - Acetaminophen'),
    ('antiemetic',                     'Antiemetic'),
    ('antibiotic',                     'Antibiotic'),
    ('anti-infective',                 'Antibiotic'),
    ('antimicrobial',                  'Antibiotic'),
    ('benzodiazepine',                 'Benzodiazepine - Sedative/Anxiolytic'),
    ('sedative',                       'Benzodiazepine - Sedative/Anxiolytic'),
    ('anxiolytic',                     'Benzodiazepine - Sedative/Anxiolytic'),
    ('anticoagulant',                  'Anticoagulant'),
    ('thrombin inhibitor',             'Anticoagulant'),
    ('factor xa',                      'Anticoagulant'),
    ('beta blocker',                   'Beta Blocker'),
    ('beta-adrenergic blocking',       'Beta Blocker'),
    ('ace inhibitor',                  'ACE Inhibitor'),
    ('angiotensin converting enzyme',  'ACE Inhibitor'),
    ('calcium channel blocker',        'Calcium Channel Blocker'),
    ('dihydropyridine',                'Calcium Channel Blocker'),
    ('nitrate',                        'Nitrate'),
    ('antiarrhythmic',                 'Antiarrhythmic'),
    ('cardiac glycoside',              'Antiarrhythmic'),
    ('diuretic',                       'Diuretic'),
    ('corticosteroid',                 'Corticosteroid'),
    ('glucocorticoid',                 'Corticosteroid'),
    ('steroid',                        'Corticosteroid'),
    ('proton pump inhibitor',          'GI - Acid Suppression'),
    ('h2-receptor',                    'GI - Acid Suppression'),
    ('acid secretion',                 'GI - Acid Suppression'),
    ('bronchodilator',                 'Bronchodilator'),
    ('beta 2-adrenergic',              'Bronchodilator'),
    ('anticholinergic bronch',         'Bronchodilator'),
    ('insulin',                        'Insulin/Glucose'),
    ('antidiabetic',                   'Insulin/Glucose'),
    ('sulfonylurea',                   'Insulin/Glucose'),
    ('antipsychotic',                  'Antipsychotic'),
    ('anticonvulsant',                 'Anticonvulsant'),
    ('gaba analog',                    'Anticonvulsant'),
    ('antiepileptic',                  'Anticonvulsant'),
    ('sodium chloride',                'IV Fluid'),
    ('iv solution',                    'IV Fluid'),
    ('electrolyte replacement',        'IV Fluid'),
    ('antiplatelet',                   'Antiplatelet'),
    ('platelet aggregation inhibitor', 'Antiplatelet'),
    ('salicylate',                     'Antiplatelet'),
    # Chronic maintenance drugs, intentionally mapped to Other
    ('statin',                         'Other'),
    ('hmg-coa reductase',              'Other'),
    ('thyroid',                        'Other'),
    ('vitamin',                        'Other'),
    ('supplement',                     'Other'),
    ('laxative',                       'Other'),
    ('antidepressant',                 'Other'),
]

def map_etcdesc_to_class(etcdesc):
    if pd.isna(etcdesc):
        return 'Other'
    val = str(etcdesc).lower()
    for pattern, cls in ETCDESC_TO_CLASS:
        if pattern in val:
            return cls
    return 'Other'

df_medrecon['shared_class'] = df_medrecon['etcdescription'].apply(map_etcdesc_to_class)

print('Shared class distribution:')
print(df_medrecon['shared_class'].value_counts())
other_pct = (df_medrecon['shared_class'] == 'Other').mean() * 100
print(f"Records mapped to 'Other': {(df_medrecon['shared_class'] == 'Other').sum():,} ({other_pct:.1f}%)")

# Diagnostic: top etcdescriptions falling into Other — candidates to add to the mapping
print('\nTop 25 etcdescriptions mapping to Other:')
print(df_medrecon[df_medrecon['shared_class'] == 'Other']['etcdescription'].value_counts().head(25))

In [ ]:
# 2. Build binary drug class feature matrix
#
# One row per ED visit 
# One binary column per drug class.
#
# Why binary not count:
#   The clinically relevant signal is presence/absence of a drug class, not how many
#   individual drugs within it. A patient on 3 beta blockers carries the same cardiac
#   risk signal as one on 1. Binary flags are also memory-efficient (uint8 = 1 byte
#   vs 8 bytes for float64) and directly interpretable.
#
# Low variance dropping:
#   Drug class columns present in fewer than 1% of visits are dropped. The model
#   sees almost no positive examples to learn a signal from.
#   recon_other is also dropped explicitly: 48.5% prevalence but near-constant
#   (fires for nearly every patient with any medications), so it adds no discriminative
#   signal despite passing the prevalence threshold.

LOW_VARIANCE_THRESHOLD = 0.01

def safe_col(s):
    return (str(s).lower()
            .replace(' ', '_').replace('-', '_').replace('/', '_')
            .replace('(', '').replace(')', '').replace(',', '')
            .replace('&', 'and')[:60])

# Deduplicate to unique (visit, class) pairs before pivoting
pairs = (df_medrecon[['ed_stay_id', 'shared_class']]
         .drop_duplicates()
         .assign(flag=1))

recon_pivot = (pairs
               .set_index(['ed_stay_id', 'shared_class'])['flag']
               .unstack(fill_value=0)
               .astype('uint8'))
recon_pivot.columns = [f'recon_{safe_col(c)}' for c in recon_pivot.columns]
recon_pivot = recon_pivot.reset_index()

recon_n = (df_medrecon.groupby('ed_stay_id')
           .agg(recon_n_total_meds = ('name', 'count'),
                recon_n_drug_classes = ('shared_class', 'nunique'))
           .reset_index())

flag_cols = [c for c in recon_pivot.columns if c.startswith('recon_')]
prevalence = recon_pivot[flag_cols].mean()
drop_cols = prevalence[prevalence < LOW_VARIANCE_THRESHOLD].index.tolist()
if 'recon_other' in recon_pivot.columns:
    drop_cols.append('recon_other')
recon_pivot = recon_pivot.drop(columns = drop_cols)
print(f'Dropped {len(drop_cols)} columns (low-variance or uninformative): {drop_cols}')

recon_features = recon_pivot.merge(recon_n, on = 'ed_stay_id', how = 'left')
print(f'med_recon feature matrix: {recon_features.shape}')
print(f'RAM: {recon_features.memory_usage(deep = True).sum() / 1e6:.1f} MB')

In [ ]:
# 3. Missing value handling
# 
# 74.1% of cohort visits have med_recon records; 25.9% (103,484 visits) do not.
#
# Why zero-fill imputation:
#   Absence of a pre-visit medication record is clinically meaningful — the patient
#   most likely takes no chronic medications, which is itself a signal (younger /
#   healthier patients). Imputing a typical drug class profile would invent chronic
#   disease burden that does not exist.
#
# etcdescription null (0.39%):
#   Records with a GSN/NDC code but no ETC taxonomy label. Excluded from class
#   mapping rather than guessing from drug name. KNN imputation deferred to the
#   modeling stage (training data only, to prevent test-set leakage).

null_etc = df_medrecon['etcdescription'].isna().sum()
null_pct = df_medrecon['etcdescription'].isna().mean() * 100
print(f'etcdescription null: {null_etc:,} ({null_pct:.2f}%)')
print('Visits with no med_recon record will receive all-zero recon_* flags '
      'when merged onto the cohort in the combined pipeline.')

In [ ]:
# 4. Save
# 
# subject_id retained for patient-level joins across pipelines.

subject_map = df_medrecon[['ed_stay_id', 'subject_id']].drop_duplicates('ed_stay_id')
recon_features = subject_map.merge(recon_features, on = 'ed_stay_id', how = 'right')

recon_features.to_csv('features_med_recon_static.csv', index = False)
print(f'Saved features_med_recon_static.csv  shape: {recon_features.shape}')
print(f'Columns: {list(recon_features.columns)}')